# Managing Fields

In `listing-fields.ipynb`, you listed the tables and fields in the Austin geodatabase and read the properties of each field. Scripts also change fields: they add fields to hold new values, calculate the values, rename fields, and delete fields that are no longer needed. In this notebook, you will use geoprocessing tools to manage the fields of a feature class in your copy of the geodatabase.

🎯 What You'll Learn

* Set up a copy of the geodatabase and the list of field names to check against
* Add one field or several fields to a feature class
* Calculate field values with Python expressions and a code block
* Rename a field and change its alias
* Delete fields from a feature class

Work through each numbered section: a demo shows how something works, then a ✏️ **Practice** prompt below it gives you a chance to try it yourself.

### ⚙️ Before you start: select the kernel

Check that the kernel in the top-right corner of the notebook is set to `arcgispro-py3`. If it is not, click the kernel name (or **Select Kernel**) and select `arcgispro-py3`. If it is not listed, refer back to `jupyter-notebook-basics.ipynb`.

---

### 1. Setup: ArcPy, folders, and a copy of the geodatabase

This notebook will work on a copy of `Austin.gdb` instead of the original in `austin_data`. The next cell creates the `outputs\m4` folder, copies `Austin.gdb` into it, and sets the environment settings so that every step in this notebook refers to the copy. The original data in `austin_data` remains untouched.

The cell copies `Austin.gdb` only if the copy does not exist yet. Rerunning the cell keeps the copy and every change you made to it. To start over with a fresh copy, follow the commented-out lines in the cell.

⚠️ **Important**: If your working folder is not `C:\workspace\gist`, update `working_folder` below.

In [ ]:
# Import ArcPy and os
import arcpy
import os

# Store the path to your working folder. Update this path to match the
# working folder you created in course-data-setup.ipynb
working_folder = r"C:\workspace\gist"
print(f"working_folder: {working_folder}")

# Build the path to the source geodatabase. It is only read, never changed
source_gdb = os.path.join(working_folder, "austin_data", "Austin.gdb")
print(f"Austin.gdb exists: {arcpy.Exists(source_gdb)}")

# Build the path to the outputs folder for this module, and create it if
# it does not exist yet
output_folder = os.path.join(working_folder, "outputs", "m4")
os.makedirs(output_folder, exist_ok=True)

# Build the path to the copy for this notebook
gdb_path = os.path.join(output_folder, "Austin.gdb")
print(f"gdb_path: {gdb_path}")

# Copy Austin.gdb only if the copy does not exist yet, so rerunning this
# cell keeps the changes you made to the copy
if not arcpy.Exists(gdb_path):
    arcpy.management.Copy(source_gdb, gdb_path)
    print("\tCopied Austin.gdb")
else:
    print("\tThe copy already exists")

    # To start over with a fresh copy, uncomment the next three lines and
    # rerun this cell. This deletes the copy and every change you made to
    # it, then copies Austin.gdb again
    # arcpy.management.Delete(gdb_path)
    # arcpy.management.Copy(source_gdb, gdb_path)
    # print("\tDeleted the copy and copied Austin.gdb again")

# Set the workspace to the copy, and allow tools to replace outputs
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

⚠️ **Important**: If the Delete tool raises an exception saying the geodatabase is locked, another program is using the copy. Close ArcGIS Pro if it is open, then restart the kernel and rerun the cells in this section from the top.

The demos in this notebook change the `police_districts` feature class in your copy of `Austin.gdb`. As in `listing-fields.ipynb`, the next cell builds the full path of the feature class and a list of its field names with a list comprehension. Every section in this notebook checks this list before it changes the table.

In [ ]:
# Build the full path of the police_districts feature class in the
# Administrative feature dataset. We will use variable gdb_path from the
# cell above
police_districts_fc = os.path.join(
    gdb_path, "Administrative", "police_districts"
)
print(f"police_districts_fc: {police_districts_fc}")

# Build the list of field names with a list comprehension. Every section
# checks this list before changing the table
pd_names = [
    one_field.name for one_field in arcpy.ListFields(police_districts_fc)
]
print(f"Number of fields: {len(pd_names)}")
print(f"pd_names: {pd_names}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 2. Adding fields: Add Field and Add Fields

In `listing-fields.ipynb`, you read the fields a table already has. Scripts also change them, and the first change is often a new field to hold new values. The **Add Field** tool adds one field to a table, and the **Add Fields** tool adds several at once. Both tools are in the Data Management Tools toolbox, and you have used them in the **Fields** view in ArcGIS Pro. In Python, the tools are the same, but two things are different:

* Nothing shows you the result. After each change, rebuild the list of field names from Section 1 to confirm it.
* A cell can be run more than once. A tool that changes a table does not work the same way the second time, for example when a field is already in the table. Check the list of field names before each change, so rerunning a cell is safe.

Every change in Sections 2 to 5 is made to the copy of `Austin.gdb` from Section 1. Section 5 deletes the fields that the demos add, so the copy ends with the same fields it started with.

For their syntax and parameters, see the [Add Field tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/add-field.html) and the [Add Fields tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/add-fields.html).

Before changing the table, save its field names in a separate list. Section 5 compares against this list to confirm that the table is back to its original fields. Run the next cell before any of the field tools in this notebook.

In [ ]:
# Example 1 - Save the original field names before changing the table

# We will use variable police_districts_fc from Section 1
print(f"police_districts_fc: {police_districts_fc}")

# Build the list of field names before any field tool runs. Section 5
# compares against this list to confirm the table is back to its original
# fields
original_names = [
    one_field.name for one_field in arcpy.ListFields(police_districts_fc)
]
print(f"Number of original fields: {len(original_names)}")
print(f"original_names: {original_names}")

The third parameter of Add Field, `field_type`, takes a **field type keyword**, a string that names the type of the new field. The keywords do not always match the strings that the `type` property of a `Field` object returns:

| `type` property | Field type keyword |
| --- | --- |
| `"String"` | `"TEXT"` |
| `"Integer"` | `"LONG"` |
| `"SmallInteger"` | `"SHORT"` |
| `"Double"` | `"DOUBLE"` |

In [ ]:
# Example 2 - Add one field with Add Field

# We will use variable pd_names from Section 1
print(f"pd_names: {pd_names}")

# Variable stores the name of the new field
label_field = "DIST_LABEL"

# Add the field only if it is not in the list of field names yet, so
# rerunning this cell does not try to add it a second time
if label_field not in pd_names:
    # TEXT is the field type keyword for a String field. field_length and
    # field_alias are optional, so keyword arguments name them and skip the
    # optional parameters before them
    arcpy.management.AddField(
        police_districts_fc,
        label_field,
        "TEXT",
        field_length=50,
        field_alias="District Label"
    )
    print(f"\tAdded {label_field}")
else:
    print(f"\t{label_field} is already in the table")

# The table changed, so rebuild the list of field names
pd_names = [
    one_field.name for one_field in arcpy.ListFields(police_districts_fc)
]

# Confirm that the new field is in the rebuilt list
print(f"{label_field} in pd_names: {label_field in pd_names}")

Add Fields takes a list of lists. Each inner list describes one field, and the position of each item decides what it means: the field name first, then the field type, then optional items such as the alias and the length. Only the name and the type are required.

In [ ]:
# Example 3 - Add several fields with Add Fields

# Each inner list describes one field, in this order:
#   [field name, field type, field alias, field length, ...]
# Only the name and type are required. The position of each item decides
# what it means, so an alias in the wrong position becomes a length
new_fields = [
    ["AREA_SQMI", "DOUBLE", "Area (sq mi)"],
    ["DIST_YEAR", "SHORT", "Review Year"]
]
print(f"new_fields: {new_fields}")

# Keep only the fields that are not in the table yet
fields_to_add = []
for one_field in new_fields:
    # one_field[0] is the field name, the first item in the inner list
    if one_field[0] not in pd_names:
        fields_to_add.append(one_field)
print(f"fields_to_add: {fields_to_add}")

# Run Add Fields only if there is at least one field to add
if len(fields_to_add) > 0:
    arcpy.management.AddFields(police_districts_fc, fields_to_add)
    print(f"\tAdded {len(fields_to_add)} field(s)")
else:
    print("\tAll of these fields are already in the table")

# The table changed, so rebuild the list of field names
pd_names = [
    one_field.name for one_field in arcpy.ListFields(police_districts_fc)
]

# Confirm that each new field is in the rebuilt list
for one_field in new_fields:
    print(f"\t{one_field[0]} in pd_names: {one_field[0] in pd_names}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 2a] Create a variable to store the full path of a feature class of your
# choice in Austin.gdb, other than police_districts and watersheds. Use a
# list comprehension to store the names of its fields, and a second list
# comprehension to store the same names as your original list. Print the
# number of fields.


# 2b] Create a variable to store your initials in capital letters. Use an
# f-string to build a TEXT field name that includes your initials, such as
# "YK_NOTE". Use Add Field to add it, with a length and an alias, only if it
# is not in your list from 2a. Rebuild your list and print whether the
# field is in it.


# 2c] Use Add Fields to add one DOUBLE field and one SHORT field, each with
# your initials in its name. Give the DOUBLE field an alias. Add only the
# fields that are not in your list from 2a. Rebuild your list and print
# whether each field is in it.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 3. Calculating field values: Calculate Field

The fields added in Section 2 are empty. The **Calculate Field** tool, in the Data Management Tools toolbox, writes a value to a field for every record in a table at once. Its third parameter is an **expression**, the calculation that produces each record's value. Its fourth parameter, `expression_type`, names the language the expression is written in: `"PYTHON3"`, `"ARCADE"`, or `"SQL"`. This course uses `"PYTHON3"`.

The expression is Python code stored in a Python string, so a value inside it needs its own quotes:

* A text value goes inside double quotes, and the whole expression goes inside single quotes: `'"Unreviewed"'`. This is the same idea as a string value inside a where clause.
* A number needs no quotes inside the expression: `"2026"`.
* In a Python expression, a field name goes between exclamation points, such as `!AREA_SQMI!`. The tool replaces it with that field's value for each record.

For its syntax and parameters, see the [Calculate Field tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/calculate-field.html).

In [ ]:
# Example 1 - Calculate fields to constant values

# We will use variable label_field from Section 2
print(f"label_field: {label_field}")

# The outer single quotes belong to the Python string, and the inner
# double quotes make Unreviewed a text value inside the expression
label_expression = '"Unreviewed"'
print(f"label_expression: {label_expression}")

# Write the value of the expression to the field for every record
arcpy.management.CalculateField(
    police_districts_fc,
    label_field,
    label_expression,
    expression_type="PYTHON3"
)
print(f"\tCalculated {label_field}")

# A number needs no quotes inside the expression
arcpy.management.CalculateField(
    police_districts_fc,
    "DIST_YEAR",
    "2026",
    expression_type="PYTHON3"
)
print("\tCalculated DIST_YEAR")

An expression can also read the shape of each record. `!shape.area@squaremiles!` passes in the area of each record's shape, converted to square miles.

In [ ]:
# Example 2 - Calculate a field from the shape of each record

# The expression returns the area of each district in square miles
area_expression = "!shape.area@squaremiles!"
print(f"area_expression: {area_expression}")

# Write each district's area to the AREA_SQMI field from Section 2
arcpy.management.CalculateField(
    police_districts_fc,
    "AREA_SQMI",
    area_expression,
    expression_type="PYTHON3"
)
print("\tCalculated AREA_SQMI")

A calculation that needs more than one line, such as an `if` statement, goes in a **code block**, a string that stores Python code for the expression to use. A code block usually defines a function, the same way you defined functions in the Functions notebook. The expression then calls the function, and Calculate Field runs it once for each record. The code block spans several lines, so it is stored in a triple-quoted string.

In [ ]:
# Example 3 - Calculate a field with a code block

# The code block is a function stored in a triple-quoted string. It takes
# the area of one district in square miles and returns a size category
size_code_block = """
def size_class(area_sqmi):
    if area_sqmi < 10:
        return "Small"
    elif area_sqmi < 30:
        return "Medium"
    else:
        return "Large"
"""

# The expression calls the function once for each record, passing in
# that record's value from the AREA_SQMI field calculated in Example 2
size_expression = "size_class(!AREA_SQMI!)"
print(f"size_expression: {size_expression}")

# The new values replace the "Unreviewed" values from Example 1.
# code_block comes after expression_type, so a keyword argument names it
arcpy.management.CalculateField(
    police_districts_fc,
    label_field,
    size_expression,
    expression_type="PYTHON3",
    code_block=size_code_block
)
print(f"\tCalculated {label_field}")

Older code, and a lot of AI-generated code, uses `"PYTHON_9.3"` or `"PYTHON"` as the expression type. Recognize these when you see them, but use `"PYTHON3"` in your code.

⚠️ **Important**: To see the calculated values, open the attribute table of `police_districts` in the copy of `Austin.gdb` in ArcGIS Pro. An open attribute table locks the feature class, and tools that add or delete fields raise an exception while it is locked. Close the attribute table, or remove the layer from the map, before running the cells in the next sections.

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 3a] Use Calculate Field to write a text value that includes your first
# name to your TEXT field from 2b. Print the expression before using it.


# 3b] Use Calculate Field to write a whole number of your choice to your
# SHORT field from 2c.


# 3c] Use Calculate Field to write a value to your DOUBLE field from 2c,
# using an expression that reads an existing numeric field in your feature
# class, such as one in your list from 2a.


# 3d] Write a code block with a function that classifies the value of your
# DOUBLE field into two or three categories of your choice. Use Calculate
# Field with your code block to write the categories to your TEXT field.


# 3e] Open the attribute table of your feature class in ArcGIS Pro to check
# the values from 3a to 3d, then close the attribute table.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 4. Renaming a field and changing its alias: Alter Field

In `listing-fields.ipynb`, you read the `name` and `aliasName` properties of each `Field` object. The **Alter Field** tool, in the Data Management Tools toolbox, changes them: it renames a field and changes its alias. Watch for the following:

* Esri's documentation does not say what happens to the alias when you rename a field, so pass `new_field_alias` every time you rename, and print both properties to confirm.
* After a rename, the old name no longer exists, so check that it is `in` the list of field names before renaming. This keeps a rerun of the cell safe.
* The field type can only be changed while the table has no records, and a TEXT field's length can only be increased once the table has records.

For its syntax and parameters, see the [Alter Field tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/alter-field-properties.html).

In [ ]:
# Example 1 - Rename a field and change its alias

# Variables store the current name, the new name, and the new alias
old_name = "AREA_SQMI"
new_name = "DIST_SQMI"
new_alias = "District Area (sq mi)"

# Rename the field only if the old name is still in the table. After the
# first run, the old name is gone, so a rerun skips the tool
if old_name in pd_names:
    # new_field_name and new_field_alias are optional, so keyword arguments
    # name them
    arcpy.management.AlterField(
        police_districts_fc,
        old_name,
        new_field_name=new_name,
        new_field_alias=new_alias
    )
    print(f"\tRenamed {old_name} to {new_name}")
else:
    print(f"\t{old_name} is not in the table")

# The table changed, so rebuild the list of field names
pd_names = [
    one_field.name for one_field in arcpy.ListFields(police_districts_fc)
]
print(f"{old_name} in pd_names: {old_name in pd_names}")
print(f"{new_name} in pd_names: {new_name in pd_names}")

# Print the name and alias of the renamed field to confirm both changed
for one_field in arcpy.ListFields(police_districts_fc, new_name):
    print(f"\tName: {one_field.name} | Alias: {one_field.aliasName}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 4a] Use Alter Field to rename your DOUBLE field from 2c and give it a new
# alias, only if its current name is in your list from 2a. Rebuild your
# list, and print the name and alias of the renamed field.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 5. Deleting fields: Delete Field

Fields that a script added for a step that is finished make a table harder to read. The **Delete Field** tool, in the Data Management Tools toolbox, removes fields from a table. A deleted field cannot be restored, except by starting over with a fresh copy of the geodatabase. Watch for the following:

* The second parameter, `drop_field`, takes one field name or a list of field names, so several fields can be deleted in one call.
* Use each field's current name. A field renamed with Alter Field is deleted by its new name.
* The optional `method` parameter set to `"KEEP_FIELDS"` deletes every field that is not in the list. Do not use it in this notebook.

For its syntax and parameters, see the [Delete Field tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/delete-field.html).

In [ ]:
# Example 1 - Delete the fields the demos added

# The current names of the fields added in Section 2. AREA_SQMI was
# renamed to DIST_SQMI in Section 4
demo_fields = ["DIST_LABEL", "DIST_SQMI", "DIST_YEAR"]

# Keep only the fields that are still in the table, so a rerun of this
# cell does not try to delete a field that is already gone
fields_to_delete = []
for one_name in demo_fields:
    if one_name in pd_names:
        fields_to_delete.append(one_name)
print(f"fields_to_delete: {fields_to_delete}")

# Run Delete Field only if there is at least one field to delete
if len(fields_to_delete) > 0:
    arcpy.management.DeleteField(police_districts_fc, fields_to_delete)
    print(f"\tDeleted {len(fields_to_delete)} field(s)")
else:
    print("\tNone of these fields are in the table")

# The table changed, so rebuild the list of field names
pd_names = [
    one_field.name for one_field in arcpy.ListFields(police_districts_fc)
]

# We will use variable original_names from Section 2. The two lists match
# when the table is back to its original fields
print(f"Back to the original fields: {pd_names == original_names}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 5a] Use one Delete Field call to delete the three fields you added in
# Section 2, using their current names after 4a. Delete only the fields
# that are in your list from 2a.


# 5b] Rebuild your list from 2a, and use == to print whether it matches
# your original list from 2a.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*